# Minecraftism - train your own uncensored Minecraft + everyday LLM (Kaggle, GPU T4 x2)

**Minecraftism** is a small open model fine-tuned into **an everyday assistant that knows Minecraft inside out** - and,
inside [PyMC Bot](https://github.com/codero-sus/PyMC_Bot), the brain that plays the game (JSON actions, chat, mob facts).

| Edition | Base (uncensored = abliterated copy by huihui-ai) | What it does |
| --- | --- | --- |
| `minecraftism` | Qwen3-4B-Instruct-2507 | text, answers straight away - **recommended** |
| `minecraftism-thinking` | Qwen3-4B-Thinking-2507 | text, thinks first (`<think>`), better at tricky questions, slower |
| `minecraftism-vl` | Qwen3-VL-4B-Instruct | also reads pictures (your screenshots and/or streamed ones) |
| `minecraftism-vl-thinking` | Qwen3-VL-4B-Thinking | pictures + thinking (slowest) |

**Uncensored** (`UNCENSORED = True`): trains on the *abliterated* base (the refusal direction removed from the weights),
drops refusals / "as an AI language model" answers from the everyday chat data and adds frank example answers, so it
answers directly instead of lecturing. The files get `-Uncensored` in their names.

**Commercial-use data** (`COMMERCIAL_ONLY = True`, the default): every training source allows commercial use -
everyday chat from OpenAssistant (Apache-2.0), `<think>` traces from NVIDIA Nemotron (CC BY 4.0), Minecraft Q&A
generated from PrismarineJS/minecraft-data (MIT) and, for the VL editions, Minecraft frames from
shinohararin/minecraft-raytrace (MIT) with exact questions and answers made from their raycast labels. The kit's
Minecraft Wiki rows (CC BY-NC-SA) are left out. The model card lists every source with its licence.
`COMMERCIAL_ONLY = False` uses the bigger non-commercial sets instead (FineTome, naklecha's 695k Minecraft Q&A,
CraftJarvis captioned screenshots, Minecraft Wiki text).

**Big data without big downloads:** the notebook streams everything external - only the rows and pictures that are
used are downloaded (minecraft-raytrace is read one ~330 MB shard at a time, without its voxel dumps; the 106 GB JARVIS set is
not used: it teaches raw keyboard/mouse tokens, not answers).

**What this notebook does**, per edition: dry-run check -> download extra everyday chat -> QLoRA training -> eval on held-out
rows -> *optional* test suite -> safetensors export (compacted: 16-bit, tied embeddings, so a 4B model stays 4B-sized)
-> **upload the safetensors** -> **every** GGUF quant made from them with llama.cpp (F16 and BF16 down to IQ1_S, with an importance matrix
from the training data), **each uploaded as soon as it is ready** -> *optional* GGUF test in Ollama -> GGUF repo (Modelfile, card, `tokenizer.json` and
configs) -> clean up.

**Two editions of every model** go to Hugging Face, each in its own repo:

| Repo (e.g.) | Format | For |
| --- | --- | --- |
| `you/Minecraftism-4B-Uncensored-GGUF` | GGUF: **every quant** (29 files: `F16`, `BF16`, `Q8_0` ~4.3 GB ... `Q4_K_M` ~2.5 GB, default ... `IQ1_S`; + `mmproj` for VL), the imatrix, `Modelfile`, `tokenizer.json` + configs | Ollama, llama.cpp, LM Studio, **PyMC Bot** |
| `you/Minecraftism-4B-Uncensored` | **safetensors**, the LoRA merged into the base in 16-bit (~8 GB for 4B params), `tokenizer.json` + configs | transformers, vLLM, further fine-tuning, your own quantisations |

Why ~8 GB for the safetensors? Every one of the 4 billion weights takes 2 bytes in 16-bit. The trainer compacts the
merge (no float32 leftovers, no duplicate output matrix), so it is exactly the base model's size and never bigger; the
small files are the GGUF quants (Q4_K_M keeps ~4.9 bits per weight).

**Before you press Run All**
1. On your PC: `python -m pymc_bot llm build` (VL editions: add `--playtime <mod folder> --screenshots`, recorded with
   `/pymcbot screenshots on`) and upload the `llm_data` folder as a Kaggle dataset (with `images/`, without `wiki_cache`).
   No PC at hand? Skip this: with no dataset added, the notebook downloads the ready-made kit
   (`pymc-minecraft-llm-kit.zip`) from the newest PyMC Bot GitHub release. An older dataset is fine too: the notebook
   then fetches the newest trainer from the release.
2. Here: **Add Input** -> your dataset. **Settings** -> Accelerator **GPU T4 x2**, Internet **on**.
3. To publish on Hugging Face: **Add-ons -> Secrets -> Add secret** `HF_TOKEN` (a *write* token from
   huggingface.co/settings/tokens) and tick it for this notebook. It is read here, never printed.
4. Edit the settings cell below: `editions_to_be_trained = {txt, txt_think, vl, vl_think}` picks the editions (only those are
   trained, e.g. `{txt}` or `{txt, vl}`). First time? Set `QUICK_TEST = True` (about 15-25 min) to check everything, then train for real.

**Distillation - the Qwen3 recipe.** Alibaba made the small Qwen3 models strong with *strong-to-weak distillation*:
a big Qwen3 teaches the small one. `DISTILL = True` does the same with the best teachers that run on 2x T4 (same
tokenizer): Qwen3-14B (4-bit AWQ) writes the answers and Qwen3-8B grades (the abliterated Qwen3-8B does both when
`UNCENSORED`): **1. off-policy** - the teacher writes the answers to `DISTILL_ROWS` chat and
Minecraft prompts (Minecraft facts are rewritten around the exact answer, game-turn JSON stays rule-checked) and the
4B learns them; **2. on-policy** - after fine-tuning, the 4B answers prompts itself and learns the teacher's probability
for every token it wrote (reverse KL; Qwen found this beats reinforcement learning at a tenth of the GPU time).

The teacher writes its answers with **vLLM** on both T4s (`VLLM_TEACHER`, installed in its own venv in the
background; several times faster than plain transformers) and keeps them in a private Hugging Face dataset
(`TEACHER_CACHE`), so the next session reuses them. The student writes its on-policy replies 4 steps at a time,
and the eval and tests run 16 prompts per batch.

**Discord updates (optional).** In Discord: channel -> Edit Channel -> Integrations -> Webhooks -> New Webhook ->
Copy Webhook URL. On Kaggle: **Add-ons -> Secrets** -> add `discord_webhook_inform_me1` (`DISCORD_SECRET`) with that
URL. The notebook then posts when the Kaggle server comes online (with the notebook's URL, `NOTEBOOK_URL`), a server
status every `DISCORD_EVERY_MIN` minutes (GPUs, RAM, disk, training step / loss / ETA, time left - if they stop coming,
the server is gone), a **THE 12 HOURs countdown** before Kaggle's 12 h limit kills the server: 5 h, 3 h, 2 h, 1 h, 30 / 15 / 10 min, then **every minute of the last 5** and a final report 30 s before, each with everything: every edition's state, its Hugging Face links, what is still only on the server (lost with it), GPUs, RAM, disk, low disk or RAM,
when the server shuts down (the full report once more), each edition's start, when the loss stops falling, each upload with its Hugging Face URL,
every result, failure and error.
**Ping a role:** add the Kaggle secret `ping_role` (`PING_ROLE_SECRET`) with the role's ID (Discord: Developer Mode on, Server Settings -> Roles -> ... -> Copy Role ID; turn on *Allow anyone to @mention this role*). That role is pinged on the important messages: startup (setup ready), each stage (training, distilling, quantizing, uploading), every Hugging Face upload, each model ready, errors and failures, low disk, THE 12 HOURs at 5 h, 3 h, 2 h and 1 h, the close and the notebook's end. Server status and the last-hour countdown stay silent.
`DISCORD_MENTION = "<@your user ID>"` pings you on: setup ready, now training / distilling / quantizing / uploading, every model upload to Hugging Face, each model ready, THE 12 HOURs at 5 h, 3 h, 2 h, 1 h, 30 / 15 / 10 / 5 / 1 min and the 30 s final report, the close, low disk, failures and errors (server status and the 4 / 3 / 2 min updates come silently). The URL is never printed.

**Speed - both T4s busy, every minute.** A Kaggle session stops after **12 h** and the weekly GPU quota is ~30 h, so
`GPU_MODE = "parallel"` (the default) trains two editions at once, one per T4; with a single edition (or the last one
left) both T4s merge on it (DDP launched with HF Accelerate, fp16 mixed precision, ~1.6-1.9x faster), so no T4 idles.
Both T4s also share the teacher's work. `GPU_MODE = "merge"` instead puts both T4s on every edition, one edition
after another. `FAST = True` puts ~12k tokens into each GPU pass (same effective batch, fewer steps), streams the shared data
once for all editions and uploads while llama.cpp makes the GGUFs; with more editions than T4s the GGUFs are made on
the CPU while the T4 already trains the next edition. While vLLM installs and the data is checked, the
base models download and the vLLM teacher is test-started on the idle T4s (`VLLM_PREFLIGHT`): a
broken vLLM costs no training time. `MAX_HOURS = 11.75` ends training early enough that the export and
upload always finish 15 min before THE 12 HOURs, Kaggle's cut-off. For long runs use **Save Version -> Save & Run All**: it
runs in the background, keeps `/kaggle/working` as the Output and the session ends by itself when done
(`AUTO_EXIT = True` also ends an interactive session once everything is on Hugging Face).

**Reading the log:** the training cell shows one status line every ~5 min (step, loss, epoch, time per step,
time to go, session time left) instead of a line per step; everything else is in `/kaggle/working/<edition>.log`.
`STOP_WHEN_LEARNED` ends an edition early once its loss stops falling. If something fails, the files to look at (or
share) are `<edition>.log`, `vllm-install.log` and, if the teacher could not start, `vllm-teacher.log`.

**Where files go:** what is needed again stays in `/kaggle/working`: models until uploaded, reports, logs, the data kit,
`cache/` (streamed data and teacher answers for every edition) and `tools/` (llama.cpp). The throwaway files go to
`/kaggle/tmp` (~60 GB scratch, not saved): model downloads, vLLM, GGUF work files, checkpoints, locks and temp files.

Made with ❤️ by codero-sus - https://github.com/codero-sus - https://codero.is-a.dev - https://the-codero.web.app/ - personal, non-commercial use (PolyForm Noncommercial 1.0.0).

In [ ]:
# ============================== settings ==============================
# Which editions to train - only these are trained (pick any, the order does not matter):
#   txt        text, answers straight away (recommended)   -> Minecraftism-4B
#   txt_think  text, thinks first (<think>)                -> Minecraftism-4B-Thinking
#   vl         also reads pictures                         -> Minecraftism-VL-4B
#   vl_think   pictures + thinking (slowest)               -> Minecraftism-VL-4B-Thinking
# e.g. {txt}  or  {txt, vl}  or  {txt, txt_think, vl, vl_think}  (quotes work too: {"txt", "vl"}; "all" = all four).
# Older text sizes by their full name: "qwen3-4b", "qwen3-8b", "qwen3-14b" (add --split to EXTRA), "llama-3.2-3b".
txt, txt_think, vl, vl_think = "txt", "txt_think", "vl", "vl_think"
editions_to_be_trained = {txt}

DATA = "auto"            # "auto" = your Kaggle dataset (Add Input) if there is one, else the ready-made kit from the
                         # newest PyMC Bot GitHub release; or a path, e.g. "/kaggle/input/my-llm-data"
TRAINER = "auto"         # "auto" = the trainer in your dataset, or the newest one from GitHub when that one is too old
                         # for these settings; "latest" = always the newest; "dataset" = never download

UNCENSORED = True        # abliterated base + refusal-free chat data (files named ...-Uncensored)
GENERAL_CHAT = 3000      # everyday chat rows mixed in so it stays a great daily assistant (0 = Minecraft only)
                         # (thinking editions get half as many rows with real <think> traces instead)
COMMERCIAL_ONLY = True   # only data whose licence allows commercial use (licences listed in the model card):
                         # OpenAssistant chat (Apache-2.0), NVIDIA Nemotron maths traces (CC BY 4.0), Q&A generated
                         # from minecraft-data (MIT), minecraft-raytrace pictures (MIT); Minecraft Wiki rows left out.
                         # False = the bigger non-commercial sets (FineTome, naklecha 695k Q&A, CraftJarvis, wiki)
MINECRAFT_QA = 20000     # Minecraft Q&A rows. Commercial: exact facts generated from minecraft-data (recipes, tools,
                         # drops, food, enchantments, mobs, biomes - about 10k rows at most). Otherwise streamed from
                         # the 695k-row naklecha/minecraft-question-answer-700k (sampled + filtered). 0 = off
STREAM_PICTURES = 3000   # VL editions: picture rows, streamed so VL trains even without your own screenshots. 0 = off
                         # Commercial: minecraft-raytrace frames, 2 exact Q&A each (biome, time, weather, block at the
                         # crosshair, mobs), read one ~330 MB shard at a time (~3 shards). Otherwise CraftJarvis/
                         # minecraft-vlp captions / Q&A read out of its zips (~150-300 MB)
PERSONA_MIX = 0.5        # share of Minecraft Q&A rows that use SYSTEM_PROMPT (game turns keep the bot's own prompt)

# The model's default personality. It goes into the training data and the Modelfile; Ollama uses it whenever an
# app sends no system prompt of its own (PyMC Bot sends its own short prompt for game turns). Edit freely.
SYSTEM_PROMPT = """You are Minecraftism, an AI assistant made by codero-sus.

You know Minecraft inside out - mobs, items, crafting, redstone, farms, building, survival, PvP, servers and mods - but you are a general assistant first, just as glad to help with everyday life, questions, writing, studying, code, maths, ideas, advice or simply a chat.

How you work:
- Be genuinely useful. Answer what was asked with the details that matter, then stop. No filler.
- Be honest. If you don't know or aren't sure, say so. Never invent facts, numbers, versions or sources.
- Treat the user as a capable adult. Don't lecture, moralise or add warnings and disclaimers nobody asked for, and don't refuse something just because it is edgy, dark, mature or controversial.
- Have a personality: friendly, witty when it fits, with your own opinions when asked.
- Match the user: short and casual for chat, clear and structured for real tasks. Reply in the user's language.
- Use Markdown only when it helps (steps, lists, code).

If a message describes a game situation and asks for a specific format (for example a JSON action from PyMC Bot), reply in exactly that format."""

HF_REPO = "auto"         # "auto" = <your HF name>/<edition repo>, e.g. Minecraftism-4B-Uncensored-GGUF; "" = no upload;
                         # "name/repo" only when training a single edition
HF_PRIVATE = False       # True = private Hugging Face repos
HF_LORA = False          # also upload the LoRA adapter
SAFETENSORS = True       # also export + upload the full-precision safetensors edition (16-bit merged, ~8 GB) to
                         # <repo without -GGUF>, e.g. Minecraftism-4B-Uncensored, next to the GGUF repo
GGUF_QUANTS = "all"      # GGUFs made from the safetensors and uploaded one by one to the GGUF repo (only when
                         # uploading): "all" = every llama.cpp quant (29, F16 to IQ1_S, most useful first; the rest
                         # is skipped if time runs out), "standard" = q8_0,q6_k,q5_k_m,q4_k_m,q3_k_m, or a list. "" = only q4_k_m

QUICK_TEST = False       # True = 30 steps, small eval, no GGUF/safetensors/upload: checks the whole pipeline in ~10-15 min
RUN_TESTS = True         # after training: bot JSON, Minecraft Q&A, everyday prompts, refusal check, identity, vision
TEST_GGUF = False        # also install Ollama here and chat with the exported GGUF (+ ~3 min per edition)
CLEAN_AFTER_UPLOAD = True  # delete the big files once an edition is safely on Hugging Face (Kaggle has 20 GB)

# ---- speed: Kaggle gives two T4s and ~30 GPU hours a week, so both T4s work all the time
GPU_MODE = "parallel"    # "parallel" (default): one edition per T4, two editions at the same time (no sync overhead);
                         #          with only ONE edition (or the last one left) both T4s merge on it, so none idles
                         # "merge": both T4s train every edition together (DDP via HF Accelerate, fp16, ~1.7-1.9x faster;
                         #          one T4 if that fails); editions run one after another, GGUFs on the CPU meanwhile
                         # "auto": same as "parallel"
                         # "single" = one T4, one edition after another (old way)
FAST = True              # max speed: batches of ~12k tokens per GPU pass (same effective batch, fewer steps), 2 data-loader
                         # workers per GPU, 60 eval rows (not 150), Hugging Face transfers at full speed
MAX_HOURS = 11.75        # THE 12 HOURs: Kaggle kills a session 12 h after it started. Everything (training, tests,
                         # export, upload) must be done 15 min before that cliff: training stops early enough that export
                         # + upload still fit, so a long run is never lost (an edition that cannot fit waits for the next
                         # session). Discord counts down: 3 h, 2 h, 1 h, 30/15/10 min, every minute of the last 5, 30 s
AUTO_EXIT = True         # end the session when everything is done (last cell) so no GPU quota idles away

# ---- Discord: messages in your channel while it trains. Discord: channel > Edit Channel > Integrations > Webhooks >
# New Webhook > Copy Webhook URL; Kaggle: Add-ons > Secrets > add that URL under DISCORD_SECRET's name (never in here)
DISCORD = True           # server online + its URL, status, warnings before / when the server closes, each edition's
                         # start / learned / uploads (with links) / result, errors
DISCORD_SECRET = "discord_webhook_inform_me1"  # name of the Kaggle secret that holds the webhook URL
DISCORD_EVERY_MIN = 30   # a server status message every N minutes (GPUs, RAM, disk, training step / loss / ETA, time
                         # left); if they stop coming, the server is gone. 0 = only the events and warnings
NOTEBOOK_URL = ""        # this notebook's address for the messages, e.g. "https://www.kaggle.com/code/you/minecraftism"
DISCORD_MENTION = ""     # who gets pinged: "<@your user ID>" (Discord: Developer Mode on, then right-click your name >
                         # Copy User ID), "@here", or "" = no ping. Pinged: setup ready, now training / distilling /
                         # quantizing / uploading, every model upload to Hugging Face, each model ready, THE 12 HOURs
                         # (5 h, 3 h, 2 h, 1 h, 30, 15, 10, 5 and 1 min, the 30 s final report, the close), low disk,
                         # failures, errors, the session's end. Silent: server status, 4/3/2 min, small progress notes
PING_ROLE_SECRET = "ping_role"  # name of the Kaggle secret that holds the Discord role to ping on the important
                         # messages: startup, each stage (training / distilling / quantizing / uploading), every HF
                         # upload, model ready, errors, low disk, THE 12 HOURs at 5/3/2/1 h, the close and the end
                         # (silent: server status, the last-hour countdown). The role's ID, e.g. 123456789012345678, or <@&ID>
                         # (Discord: Developer Mode on, Server Settings > Roles > ... > Copy Role ID; turn on "Allow
                         # anyone to @mention this role"). No such secret = no role ping. Never printed

# ---- distillation: Qwen3's "strong-to-weak" recipe (how Alibaba made the small Qwen3 models so strong)
DISTILL = True           # a bigger Qwen3 teaches the 4B student, in two phases:
                         # 1. off-policy: the teacher writes the answers to DISTILL_ROWS chat + Minecraft prompts and
                         #    the student learns them (Minecraft facts rewritten around the exact answer; game turns kept)
                         # 2. on-policy: after fine-tuning, the student answers prompts itself and learns the teacher's
                         #    probability for every token it wrote (reverse KL; text editions)
TEACHER = "auto"         # "auto" = the best teacher for each phase on 2x T4: Qwen3-14B (4-bit AWQ, on vLLM) writes the
                         # answers, Qwen3-8B (fits next to the student) grades; UNCENSORED: abliterated Qwen3-8B for both.
                         # A name, e.g. "unsloth/Qwen3-14B", is used for both phases (same Qwen3 tokenizer needed)
DISTILL_ROWS = 1500      # answers the teacher writes (both T4s, ~25-40 min; cached, so the next editions reuse them)
DISTILL_MINUTES = 40     # most minutes the teacher may write; rows it doesn't reach keep their original answers
ONPOLICY_STEPS = 60      # on-policy steps, 4 replies per T4 each (~15-20 min per text edition); 0 = phase 1 only
VLLM_TEACHER = True      # the teacher writes with vLLM on both T4s (several times faster than transformers); installed
                         # in its own venv in the background while the data is checked, transformers if it can't run
VLLM_PREFLIGHT = True    # start the teacher once on vLLM while the T4s are still idle (data check): a broken vLLM is
                         # found in ~5 min of idle time instead of eating ~40 min of the training's time, and the
                         # teacher is already downloaded when the training needs it
SLOW_TEACHER = False     # when vLLM can't run: False = the teacher writes no new phase-1 answers with transformers (~10x
                         # slower, a few hundred answers in 40 min) - training starts at once with the cached answers;
                         # phase 2 (on-policy) still runs. True = transformers writes for up to DISTILL_MINUTES
TEACHER_CACHE = True     # keep the teacher's answers in a private HF dataset (<you>/Minecraftism-teacher-answers):
                         # the next session reuses them instead of writing them again (needs HF_TOKEN)

EPOCHS = 1               # 2 = learns the data better, up to twice as long (STOP_WHEN_LEARNED cuts it short once learned)
STOP_WHEN_LEARNED = True  # end an edition once its loss stops falling (templated data is learned long before
                          # the last epoch): hours of weekly GPU time saved for the next edition
EXTRA = ""               # more trainer flags, e.g. "--gguf q5_k_m", "--rank 64", "--lr 1e-4", "--image-side 448"

In [ ]:
%%capture
# Unsloth's Kaggle install (same as their Qwen3 / Qwen3-VL notebooks)
!pip install pip3-autoremove
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install unsloth
!pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.57.6
!pip install --no-deps trl==0.22.2

In [ ]:
# ============================== environment check ==============================
import glob
import importlib.util
import json
import os
import re
import shutil
import socket
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
import zipfile
from queue import Queue as DiscordQueue

# ---- where files go: /kaggle/working keeps what is needed again (models until uploaded, reports, logs, the data kit,
# the trainer, the shared data cache, llama.cpp); /kaggle/tmp gets the throwaway files (~60 GB scratch, not saved,
# not counted in the 20 GB of /kaggle/working): model downloads, vLLM, GGUF intermediates, checkpoints, locks, temp
WORK, TMP = "/kaggle/working", "/kaggle/tmp"
CACHE = f"{WORK}/cache"  # chat / Q&A / pictures / teacher answers: shared by every edition (and kept for the next
                         # session when the notebook's file persistence is on)
TOOLS = f"{WORK}/tools"  # llama.cpp, downloaded once and reused by every edition's GGUF export
for folder in (CACHE, TOOLS, f"{TMP}/hf-cache", f"{TMP}/tmp", f"{TMP}/run", f"{TMP}/locks"):
    os.makedirs(folder, exist_ok=True)
os.environ.update({
    "PYMC_SCRATCH": TMP,                  # the trainer's throwaway files (GGUF work, checkpoints, hand-over, locks)
    "HF_HUB_CACHE": f"{TMP}/hf-cache",    # downloaded base / teacher models (re-downloaded each session anyway)
    "TMPDIR": f"{TMP}/tmp",               # every tempfile
    "PYMC_LOGS": WORK,                    # a failed vLLM teacher's full log is kept here (vllm-teacher.log)
    "UV_CACHE_DIR": f"{TMP}/uv-cache", "PIP_CACHE_DIR": f"{TMP}/pip-cache",
})
CPUS = os.cpu_count() or 4  # every core works: tokenizers, data loaders and the GGUF quantizer use them all
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")
os.environ.setdefault("OMP_NUM_THREADS", str(max(1, CPUS // 2)))  # two training processes share the cores

# ---- the editions picked in editions_to_be_trained -> trainer presets, longest first (both T4s finish together)
EDITION_NAMES = {"txt": "minecraftism", "txt_think": "minecraftism-thinking", "vl": "minecraftism-vl",
                 "vl_think": "minecraftism-vl-thinking"}
OTHER_SIZES = ("qwen3-4b", "qwen3-8b", "qwen3-14b", "llama-3.2-3b", "qwen3.5-4b")


def chosen_editions(wanted):
    if isinstance(wanted, str):
        wanted = list(EDITION_NAMES) if wanted.strip().lower() == "all" else re.split(r"[,\s]+", wanted.strip())
    names = []
    for item in wanted:
        key = str(item).strip()
        name = EDITION_NAMES.get(key.lower().replace("-", "_"), key)
        if name not in EDITION_NAMES.values() and name not in OTHER_SIZES:
            raise ValueError(f"editions_to_be_trained: unknown edition {key!r} - use any of {', '.join(EDITION_NAMES)}")
        if name not in names:
            names.append(name)
    order = list(EDITION_NAMES.values())[::-1] + list(OTHER_SIZES)  # vl_think, vl, txt_think, txt: longest first
    return sorted(names, key=order.index)


EDITIONS = chosen_editions(editions_to_be_trained)
assert EDITIONS, "editions_to_be_trained is empty - pick at least one of {txt, txt_think, vl, vl_think}"
print("editions to train (longest first):", ", ".join(EDITIONS))


def check(ok, good, bad):
    print(("OK    " if ok else "WARN  ") + (good if ok else bad))
    return ok

NO_GPU = ("no GPU - this run has Accelerator: None. In the notebook editor: Settings (right panel) > Accelerator > "
          "GPU T4 x2, then Save Version again (Save & Run All uses the editor's accelerator)")
try:  # no GPU attached = no nvidia-smi at all
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True, timeout=60).stdout.strip().splitlines()
except (OSError, subprocess.SubprocessError):
    gpus = []
check(bool(gpus), f"GPU: {', '.join(gpus)}", NO_GPU)
try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
    online = True
except OSError:
    online = False
check(online, "internet is on", "internet is OFF - Settings > Internet on (needed for the models and the chat data)")
free_gb = shutil.disk_usage(WORK).free / 1e9
check(free_gb > 12, f"{free_gb:.0f} GB free in {WORK}", f"only {free_gb:.0f} GB free - clean {WORK}")
tmp_gb = shutil.disk_usage(TMP).free / 1e9  # models, vLLM and the GGUF work go here, off the 20 GB output disk
check(tmp_gb > 20, f"{tmp_gb:.0f} GB free in {TMP} for downloads and the GGUF quants", f"only {tmp_gb:.0f} GB free in {TMP}")

# ---- training data + trainer: your dataset, else the newest PyMC Bot release on GitHub
RELEASES = "https://api.github.com/repos/codero-sus/PyMC_Bot/releases?per_page=30"

def release_asset(name):
    """Download URL of `name` from the newest PyMC Bot release that has it (None when offline)."""
    try:
        request = urllib.request.Request(RELEASES, headers={"Accept": "application/vnd.github+json"})
        releases = json.load(urllib.request.urlopen(request, timeout=30))
    except Exception as exc:
        print(f"WARN  GitHub releases not reachable ({type(exc).__name__})")
        return None
    for release in sorted(releases, key=lambda r: r.get("published_at") or "", reverse=True):
        for asset in release.get("assets", []):
            if asset["name"] == name:
                return asset["browser_download_url"], release["tag_name"]
    return None

def download(name, target):
    found = release_asset(name)
    if not found:
        return None
    url, tag = found
    urllib.request.urlretrieve(url, target)
    print(f"OK    downloaded {name} from release {tag}")
    return target

if DATA == "auto":
    found = sorted(glob.glob("/kaggle/input/**/train.jsonl", recursive=True))
    if found:
        DATA = os.path.dirname(found[0])
    else:
        print("no dataset added - using the ready-made Minecraftism kit (no recorded playtime / screenshots in it)")
        kit = download("pymc-minecraft-llm-kit.zip", f"{TMP}/pymc-minecraft-llm-kit.zip")  # the zip is throwaway
        assert kit, "No data: add your llm_data dataset with Add Input, or turn Internet on to download the kit"
        with zipfile.ZipFile(kit) as archive:
            archive.extractall("/kaggle/working/kit")
        os.remove(kit)
        DATA = "/kaggle/working/kit/llm_data"
assert os.path.isfile(f"{DATA}/train.jsonl"), f"{DATA} has no train.jsonl - run: python -m pymc_bot llm build"
pictures = len(glob.glob(f"{DATA}/images/*"))
print(f"OK    data: {DATA} ({pictures} pictures)")

def trainer_is_current(path):
    text = open(path, encoding="utf-8").read()
    return all(flag in text for flag in ("--uncensored", "--system-prompt-file", "--test-only", "--general",
                                             "--stream-vision", "--minecraft-qa", "--commercial",
                                             "--safetensors", "--gguf-quants", "--deadline", "--export-lock",
                                             "--token-budget", "--defer-gguf", "--distill-rows", "--onpolicy-steps",
                                             "--vllm-python", "--tools-dir"))

SCRIPT = f"{DATA}/train_minecraft_llm.py" if os.path.isfile(f"{DATA}/train_minecraft_llm.py") else ""
if TRAINER == "latest" or (TRAINER == "auto" and not (SCRIPT and trainer_is_current(SCRIPT))):
    if SCRIPT:
        print("the trainer in your dataset is older than this notebook - fetching the newest one")
    SCRIPT = download("train_minecraft_llm.py", "/kaggle/working/train_minecraft_llm.py") or SCRIPT
assert SCRIPT, "No train_minecraft_llm.py: rebuild your data with python -m pymc_bot llm build, or turn Internet on"
check(trainer_is_current(SCRIPT), f"trainer: {SCRIPT}",
      f"trainer {SCRIPT} is old - UNCENSORED / tests / safetensors need a newer one (rebuild the data)")

# Hugging Face token from Kaggle's secrets (never printed)
if HF_REPO and not QUICK_TEST:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
        print("OK    HF_TOKEN loaded - finished editions are uploaded to Hugging Face")
    except Exception as exc:
        HF_REPO = ""
        print(f"WARN  no HF_TOKEN secret ({type(exc).__name__}) - models stay in /kaggle/working (Output tab)")

# ---- Discord: the webhook URL comes from Kaggle's secrets (never printed); messages go out in the background
DISCORD_URL, DISCORD_STATE = "", {"sent": {}, "dead": False}
DISCORD_HOOK = re.compile(r"https://(?:ptb\.|canary\.)?discord(?:app)?\.com/api/(?:v\d+/)?webhooks/\d+/[\w-]+$")
DISCORD_COLORS = {"info": 0x5865F2, "ok": 0x57F287, "warn": 0xFEE75C, "bad": 0xED4245}
DISCORD_OUTBOX = DiscordQueue()
if DISCORD:
    try:
        from kaggle_secrets import UserSecretsClient
        DISCORD_URL = (UserSecretsClient().get_secret(DISCORD_SECRET) or "").strip()
    except Exception as exc:
        print(f"WARN  no {DISCORD_SECRET} secret ({type(exc).__name__}) - no Discord messages (Add-ons > Secrets)")
    if DISCORD_URL and not DISCORD_HOOK.match(DISCORD_URL.split("?")[0]):
        DISCORD_URL = ""
        print(f"WARN  {DISCORD_SECRET} is not a Discord webhook URL (https://discord.com/api/webhooks/...) - no messages")

# the role pinged on every message: its ID from Kaggle's secrets (never printed)
DISCORD_ROLE = ""
if DISCORD_URL and PING_ROLE_SECRET:
    try:
        from kaggle_secrets import UserSecretsClient
        _role = (UserSecretsClient().get_secret(PING_ROLE_SECRET) or "").strip()
    except Exception:
        _role = ""
        print(f"INFO  no {PING_ROLE_SECRET} secret - Discord messages ping no role (Add-ons > Secrets)")
    _match = re.fullmatch(r"<@&(\d{5,25})>|@?&?(\d{5,25})", _role)
    if _match:
        DISCORD_ROLE = _match.group(1) or _match.group(2)
        print(f"OK    {PING_ROLE_SECRET} loaded - important Discord messages ping that role")
    elif _role:
        print(f"WARN  {PING_ROLE_SECRET} is not a Discord role ID (digits, or <@&ID>) - no role ping")
    del _role, _match


def discord_post(payload):
    """POST one message; waits out Discord's rate limit, gives up quietly on network trouble."""
    body = json.dumps(payload).encode("utf-8")
    for _attempt in range(4):
        request = urllib.request.Request(DISCORD_URL, data=body, method="POST", headers={
            "Content-Type": "application/json", "User-Agent": "PyMC-Bot-Kaggle (https://github.com/codero-sus/PyMC_Bot, 1)"})
        try:
            urllib.request.urlopen(request, timeout=15).close()
            return True
        except urllib.error.HTTPError as exc:
            if exc.code == 429:  # rate limited: Discord says how long to wait
                try:
                    wait = float(json.loads(exc.read() or b"{}").get("retry_after", 2))
                except (ValueError, AttributeError):
                    wait = 2.0
                time.sleep(min(max(wait, 0.5), 30))
                continue
            if exc.code in (401, 403, 404):  # webhook deleted or wrong: stop trying
                DISCORD_STATE["dead"] = True
                print(f"WARN  Discord answered HTTP {exc.code} - check the {DISCORD_SECRET} secret; no more messages",
                      flush=True)
                return False
            time.sleep(2)
        except (OSError, ValueError):
            time.sleep(2)
    return False


def discord_sender():
    while True:
        payload = DISCORD_OUTBOX.get()
        try:
            if not DISCORD_STATE["dead"]:
                discord_post(payload)
        finally:
            DISCORD_OUTBOX.task_done()


def discord(text, title="", level="info", ping=False, key=None, every=0.0, role=None):
    """A Discord message (embed) sent in the background. key + every: at most one such message per `every` seconds.
    The ping_role role hears the important ones: `ping` and errors ("bad"), unless `role` says otherwise."""
    if not DISCORD_URL or DISCORD_STATE["dead"]:
        return False
    if key is not None:
        if time.time() - DISCORD_STATE["sent"].get(key, 0.0) < every:
            return False
        DISCORD_STATE["sent"][key] = time.time()
    embed = {"description": str(text)[:4000], "color": DISCORD_COLORS.get(level, DISCORD_COLORS["info"]),
             "footer": {"text": "Minecraftism on Kaggle · PyMC Bot by codero-sus"},
             "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    if title:
        embed["title"] = str(title)[:256]
    payload = {"username": "Minecraftism trainer", "embeds": [embed],
               "avatar_url": "https://raw.githubusercontent.com/codero-sus/PyMC_Bot/main/icon.png",
               "allowed_mentions": {"parse": []}}
    role_on = bool(DISCORD_ROLE) and ((ping or level == "bad") if role is None else bool(role))
    mentions = [f"<@&{DISCORD_ROLE}>"] if role_on else []  # the ping_role role: important messages only
    if ping and DISCORD_MENTION.strip():
        if DISCORD_MENTION.strip()[:200] not in mentions:
            mentions.append(DISCORD_MENTION.strip()[:200])
        payload["allowed_mentions"] = {"parse": ["users", "roles", "everyone"]}
    elif role_on:
        payload["allowed_mentions"] = {"parse": [], "roles": [DISCORD_ROLE]}  # only that role, nothing in the text
    if mentions:
        payload["content"] = " ".join(mentions)
    DISCORD_OUTBOX.put(payload)
    return True


def discord_flush(seconds=20):
    """Wait (a little) until queued messages are out - before the session ends."""
    end = time.time() + seconds
    while DISCORD_OUTBOX.unfinished_tasks and time.time() < end:
        time.sleep(0.2)


def discord_on_error(result):
    """A cell failed: tell Discord (the session may now sit idle until you look)."""
    error = getattr(result, "error_in_exec", None) or getattr(result, "error_before_exec", None)
    if error is None:
        return
    if isinstance(error, KeyboardInterrupt):
        discord("A cell was interrupted (stopped by hand).", "Notebook interrupted", "warn")
    else:
        discord(f"`{type(error).__name__}: {str(error)[:1500]}`\nThe notebook stopped - open it on Kaggle to see the "
                "cell output.", "Notebook error", "bad", ping=True)
        discord_flush(15)  # Save & Run All shuts the kernel down right after a failed cell


if DISCORD_URL:
    threading.Thread(target=discord_sender, daemon=True).start()
    try:  # every later cell that fails reports itself
        from IPython import get_ipython
        shell = get_ipython()
        if shell is not None:
            for hook in list(shell.events.callbacks["post_run_cell"]):
                if getattr(hook, "__name__", "") == "discord_on_error":  # this cell ran before
                    shell.events.unregister("post_run_cell", hook)
            shell.events.register("post_run_cell", discord_on_error)
    except Exception as exc:
        print(f"WARN  Discord: failed cells will not be reported ({type(exc).__name__})")
    print("OK    Discord webhook loaded - progress and results go to your channel")
# ---- end Discord

if not gpus:  # stop here: hours of downloads would end in a CUDA error anyway (Discord hears why)
    raise RuntimeError(NO_GPU)

# the system prompt goes to the trainer as a file (no shell quoting problems)
SYSTEM_FILE = f"{TMP}/system_prompt.txt"  # written again every run"
with open(SYSTEM_FILE, "w", encoding="utf-8") as handle:
    handle.write(SYSTEM_PROMPT.strip())

FLAGS = f'--data "{DATA}" --system-prompt-file "{SYSTEM_FILE}" --persona-mix {PERSONA_MIX} --epochs {EPOCHS}'
FLAGS += " --uncensored" if UNCENSORED else ""
FLAGS += " --commercial" if COMMERCIAL_ONLY else ""
try:  # the session started when this kernel did
    import psutil
    SESSION_START = psutil.Process(os.getpid()).create_time()
except Exception:
    SESSION_START = time.time()
DEADLINE = SESSION_START + MAX_HOURS * 3600
FLAGS += f" --deadline {DEADLINE:.0f} --export-lock {TMP}/locks/export.lock --tools-dir {TOOLS}"
FLAGS += " --clean-after-upload" if CLEAN_AFTER_UPLOAD else ""
FLAGS += f" --stream-cache {CACHE}"  # chat / Q&A / pictures / teacher answers fetched once, used by every edition
LOADER_WORKERS = max(2, min(4, CPUS // max(1, len(gpus))))  # data loaders per T4: as many as the cores allow
FLAGS += f" --token-budget 12288 --loader-workers {LOADER_WORKERS} --eval-limit 60" if FAST else ""
FLAGS += " --stop-when-learned" if STOP_WHEN_LEARNED else ""
if FAST:  # hf_xet's high-performance mode: model downloads and uploads use all cores and the full bandwidth
    os.environ.setdefault("HF_XET_HIGH_PERFORMANCE", "1")
if QUICK_TEST:
    FLAGS += f" --max-steps 30 --eval-limit 10 --gguf none --safetensors none --general {min(GENERAL_CHAT, 200)}"
    FLAGS += f" --minecraft-qa {min(MINECRAFT_QA, 200)} --stream-vision {min(STREAM_PICTURES, 40)}"
else:
    FLAGS += f" --general {GENERAL_CHAT} --minecraft-qa {MINECRAFT_QA} --stream-vision {STREAM_PICTURES}"
    FLAGS += " --safetensors 16bit" if SAFETENSORS else " --safetensors none"
    if HF_REPO:
        FLAGS += f' --hub-repo "{HF_REPO}"' + (" --hub-private" if HF_PRIVATE else "") + (" --hub-lora" if HF_LORA else "")
        if str(GGUF_QUANTS).strip().lower() in ("all", "standard") and "--imatrix-chunks" not in open(SCRIPT, encoding="utf-8").read():
            print(f'WARN  this trainer predates GGUF_QUANTS = "{GGUF_QUANTS}" - it makes the standard 5 quants '
                  "(the next PyMC Bot release brings the one that makes every quant)")
            GGUF_QUANTS = "q8_0,q6_k,q5_k_m,q4_k_m,q3_k_m"
        FLAGS += f' --gguf-quants "{GGUF_QUANTS or "none"}"'
    else:
        FLAGS += " --gguf-quants none"  # without an upload only the main GGUF is kept (Kaggle's 20 GB)
    if TEST_GGUF:
        FLAGS += " --test-gguf"
if DISTILL:  # QUICK_TEST: a few teacher answers + 3 on-policy steps, just to check that phase works
    FLAGS += f' --teacher "{TEACHER}" --distill-minutes {DISTILL_MINUTES}'
    FLAGS += " --onpolicy-refresh 4" if FAST else ""  # replies for 4 steps in one batched call
    FLAGS += " --teacher-cache-repo auto" if TEACHER_CACHE and HF_REPO else ""
    FLAGS += f" --distill-rows {min(DISTILL_ROWS, 32) if QUICK_TEST else DISTILL_ROWS}"
    FLAGS += f" --onpolicy-steps {min(ONPOLICY_STEPS, 3) if QUICK_TEST else ONPOLICY_STEPS}"
FLAGS += " --test" if RUN_TESTS else ""
FLAGS += f" {EXTRA}"
print("trainer flags:", FLAGS)

# ---- the trainer's own presets and teacher choice (so the prefetch and the pre-flight pick the models it will use)
def load_trainer_module(path):
    try:
        spec = importlib.util.spec_from_file_location("pymc_trainer", path)
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        return module
    except Exception as exc:
        print(f"WARN  could not read the trainer's presets ({type(exc).__name__}) - no prefetch")
        return None


TRAINER_MODULE = load_trainer_module(SCRIPT)


def trainer_args(**extra):
    import argparse
    values = {"teacher": TEACHER, "uncensored": UNCENSORED, "teacher_engine": "auto", "vllm_python": ""}
    values.update(extra)
    return argparse.Namespace(**values)


# ---- prefetch: the first editions' base models + the grading teacher download now, while vLLM installs and the data
# is checked (the network idles then), instead of after the T4s are already waiting for them
PREFETCH_SKIP = ["*.gguf", "*.bin", "*.pt", "*.pth", "*.onnx", "original/*", "onnx/*", "*.msgpack", "*.h5"]


def models_to_prefetch():
    if TRAINER_MODULE is None:
        return []
    repos = []
    for number, name in enumerate(EDITIONS[:2]):  # the editions that start first (one per T4)
        preset = TRAINER_MODULE.PRESETS.get(name)
        if preset:
            preset = TRAINER_MODULE.uncensored_preset(name, preset) if UNCENSORED else preset
            repos.append(preset["model"])
        if number == 0 and DISTILL and ONPOLICY_STEPS:
            repos.append(TRAINER_MODULE.teacher_model(trainer_args(), "grade"))
    return list(dict.fromkeys(repos))


def prefetch_models(repos, download=None):
    if download is None:
        try:
            from huggingface_hub import snapshot_download as download
        except ImportError:
            return []
    done = []
    for repo in repos:
        if shutil.disk_usage(TMP).free / 1e9 < 34:  # leave room for vLLM, its teacher and the GGUF work
            print(f"WARN  prefetch stopped: under 34 GB free in {TMP} ({repo} downloads when it is needed)")
            break
        started = time.time()
        try:
            download(repo, ignore_patterns=PREFETCH_SKIP, max_workers=16)
            done.append(repo)
            print(f"OK    prefetched {repo} in {time.time() - started:.0f} s", flush=True)
        except Exception as exc:  # the trainer downloads it itself
            print(f"WARN  prefetch of {repo} failed ({type(exc).__name__}) - it downloads when it is needed", flush=True)
    return done


if FAST and online and not globals().get("PREFETCH_THREAD"):
    PREFETCH = models_to_prefetch()
    if PREFETCH:
        PREFETCH_THREAD = threading.Thread(target=prefetch_models, args=(PREFETCH,), daemon=True)
        PREFETCH_THREAD.start()
        print(f"OK    downloading {', '.join(PREFETCH)} in the background")
# ---- end prefetch

# ---- vLLM for the distillation teacher: its own venv (Unsloth's torch stays as it is), installed in the background
VLLM_VERSION = "0.18.1"  # validated on Kaggle's 2x T4 (torch 2.10, CUDA 12.8)
VLLM_PYTHON = f"{TMP}/vllm/bin/python"
VLLM_READY = threading.Event()
VLLM_ABORT = threading.Event()  # the training can't wait any longer: the pre-flight stops and frees the T4s at once
VLLM_STATE = {"proc": None, "cause": "", "teacher": ""}
VLLM_WAIT = 20 * 60  # the training cell waits at most this long for the install + pre-flight
VLLM_PREFLIGHT_MIN = 14  # a pre-flight that takes longer is a hang


def install_vllm():
    def step(command, log):
        return subprocess.run(command, stdout=log, stderr=subprocess.STDOUT).returncode == 0

    with open(f"{WORK}/vllm-install.log", "w", encoding="utf-8") as log:
        if not os.path.exists(VLLM_PYTHON):
            fast = (step([sys.executable, "-m", "pip", "install", "-q", "uv"], log)
                    and step([sys.executable, "-m", "uv", "venv", "-q", f"{TMP}/vllm", "--python", sys.executable], log)
                    and step([sys.executable, "-m", "uv", "pip", "install", "-q", "--python", VLLM_PYTHON,
                              f"vllm=={VLLM_VERSION}"], log))
            if not fast:  # plain pip (slower)
                shutil.rmtree(f"{TMP}/vllm", ignore_errors=True)
                if not (step([sys.executable, "-m", "venv", f"{TMP}/vllm"], log)
                        and step([VLLM_PYTHON, "-m", "pip", "install", "-q", f"vllm=={VLLM_VERSION}"], log)):
                    VLLM_STATE["cause"] = f"vLLM did not install - see {WORK}/vllm-install.log"
                    return
        if not step([VLLM_PYTHON, "-c", "import vllm, torch; assert torch.cuda.is_available()"], log):
            VLLM_STATE["cause"] = f"vLLM does not import - see {WORK}/vllm-install.log"
            return
    if not VLLM_PREFLIGHT or VLLM_ABORT.is_set() or vllm_preflight():
        if not VLLM_ABORT.is_set():
            VLLM_READY.set()


# ---- vLLM pre-flight: the teacher starts once on the idle T4s with the trainer's own settings and answers one prompt
VLLM_PREFLIGHT_CODE = """import sys, time
from vllm import LLM, SamplingParams
started = time.time()
options = dict(model=sys.argv[1], dtype="float16", tensor_parallel_size=int(sys.argv[2]), max_model_len=4608,
               gpu_memory_utilization=0.88, enforce_eager=True, disable_custom_all_reduce=True,
               enable_prefix_caching=True)
if sys.argv[3]:
    options["quantization"] = sys.argv[3]
llm = LLM(**options)
reply = llm.generate(["Say OK."], SamplingParams(max_tokens=8), use_tqdm=False)[0].outputs[0].text
print("PREFLIGHT OK", round(time.time() - started), repr(reply), flush=True)
"""


def preflight_teacher():
    """The model the trainer's teacher writes the phase-1 answers with."""
    if TRAINER_MODULE is not None:
        return TRAINER_MODULE.teacher_model(trainer_args(vllm_python=VLLM_PYTHON), "answers")
    if TEACHER.strip().lower() != "auto":
        return TEACHER.strip()
    return "huihui-ai/Huihui-Qwen3-8B-abliterated-v2" if UNCENSORED else "Qwen/Qwen3-14B-AWQ"


def stop_process_group(proc):
    import signal
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (OSError, AttributeError):
            pass
        try:
            proc.wait(timeout=20)
            return
        except Exception:
            continue


def vllm_preflight(popen=None, timeout=None):
    """True when vLLM runs the teacher on this machine. A failure says why (vllm-preflight.log keeps the rest)."""
    popen = popen or subprocess.Popen
    timeout = VLLM_PREFLIGHT_MIN * 60 if timeout is None else timeout
    teacher = VLLM_STATE["teacher"] = preflight_teacher()
    split = 2 if len(gpus) >= 2 else 1
    quant = "" if split >= 2 or re.search(r"(?i)[-_.](awq|gptq)(?:$|[-_.])", teacher) else "bitsandbytes"
    path, started = f"{WORK}/vllm-preflight.log", time.time()
    print(f"OK    vLLM pre-flight: starting the teacher {teacher} on {split} idle T4(s)...", flush=True)
    with open(path, "w", encoding="utf-8") as log:
        proc = popen([VLLM_PYTHON, "-c", VLLM_PREFLIGHT_CODE, teacher, str(split), quant], stdout=log,
                     stderr=subprocess.STDOUT, env={**os.environ, "VLLM_LOGGING_LEVEL": "INFO"}, start_new_session=True)
        VLLM_STATE["proc"] = proc
        while proc.poll() is None and time.time() - started < timeout and not VLLM_ABORT.is_set():
            time.sleep(2)
        hung = proc.poll() is None
        stop_process_group(proc)  # the engine's GPU processes too: the T4s must be empty for the training
    with open(path, encoding="utf-8", errors="replace") as log:
        text = log.read()
    took = time.time() - started
    if "PREFLIGHT OK" in text:
        print(f"OK    vLLM pre-flight: the teacher answered after {took / 60:.1f} min - vLLM writes the answers",
              flush=True)
        discord(f"vLLM runs the teacher `{teacher}` ({took / 60:.1f} min to start, now downloaded).", "Teacher ready",
                "ok")
        return True
    if VLLM_ABORT.is_set():
        VLLM_STATE["cause"] = "the pre-flight took too long - the training could not wait any more"
        return False
    found = TRAINER_MODULE.vllm_root_cause(text) if TRAINER_MODULE is not None else ""
    VLLM_STATE["cause"] = (f"engine not ready after {took / 60:.0f} min (a hang)" if hung else
                           (found or text[-800:]).strip() or "no output")
    print(f"WARN  vLLM pre-flight failed after {took / 60:.1f} min - cause:\n{VLLM_STATE['cause']}\n"
          f"      (full log: {path})", flush=True)
    discord(f"vLLM could not run the teacher `{teacher}` (found in the pre-flight, after {took / 60:.1f} min - "
            "no training time lost). Cause:\n```\n" + VLLM_STATE["cause"][-1200:].replace("```", "` ` `")
            + f"\n```\nFull log: `{path}`", "vLLM pre-flight failed", "warn")
    return False
# ---- end vLLM pre-flight


if DISTILL and VLLM_TEACHER and online:
    if shutil.disk_usage(TMP).free / 1e9 > 40:
        VLLM_THREAD = threading.Thread(target=install_vllm, daemon=True)
        VLLM_THREAD.start()
        print(f"OK    installing vLLM {VLLM_VERSION} for the teacher in the background ({TMP}/vllm)"
              + (", then a pre-flight on the idle T4s" if VLLM_PREFLIGHT else ""))
    else:
        print(f"WARN  under 40 GB free in {TMP} - the teacher uses transformers instead of vLLM (slower)")


# ---- Discord server monitor: online message with the URL, status every DISCORD_EVERY_MIN, warnings before it closes
KAGGLE_LIMIT = SESSION_START + 12 * 3600  # Kaggle closes every session 12 h after it started
SERVER_URL = NOTEBOOK_URL.strip() or "https://www.kaggle.com/code (Your Work)"


def folder_gb(path):
    total = 0
    for root, _dirs, files in os.walk(path):
        for name in files:
            try:
                total += os.path.getsize(os.path.join(root, name))
            except OSError:
                pass
    return total / 1e9


def edition_lines():
    """Each edition: its state, its Hugging Face links (safe) and what would be lost with the server."""
    lines, safe = [], 0
    codes, read = globals().get("CODES", {}), globals().get("report_of")
    status = globals().get("DISCORD_STATUS", {})
    for edition in globals().get("READY") or EDITIONS:
        try:
            info = read(edition) if read else {}
        except Exception:
            info = {}
        code, links = codes.get(edition), []
        if info.get("safetensors_uploaded") and info.get("hub_repo_safetensors"):
            links.append(f"[safetensors](https://huggingface.co/{info['hub_repo_safetensors']})")
        if info.get("hub_repo") and code == 0:
            links.append(f"[GGUF](https://huggingface.co/{info['hub_repo']})")
        if code == 0:
            state = "✅ done"
        elif code == 3:
            state = "⏭️ skipped - no time left, train it next session"
        elif code is not None:
            state = f"❌ failed (exit {code})"
        elif info:
            state = "📦 exporting / uploading"
        elif edition in status:
            state = "🏋️ " + status[edition]
        else:
            state = "⏳ waiting for a T4"
        safe += bool(links)
        local = folder_gb(f"{WORK}/{edition}") if not links else 0.0
        lines.append(f"**{edition}:** {state}" + (" · " + " · ".join(links) if links else "")
                     + (f" · ⚠️ {local:.1f} GB only in /kaggle/working/{edition}" if local >= 0.05 else ""))
    return lines, safe


def server_status(quick=False):
    """GPUs, RAM, disk, every edition and time left - one Discord message (quick: no nvidia-smi call)."""
    now = time.time()
    lines = [f"**Server:** {SERVER_URL}",
             f"**Up** {(now - SESSION_START) / 3600:.1f} h · THE 12 HOURs (Kaggle closes it) <t:{int(KAGGLE_LIMIT)}:R> "
             f"(<t:{int(KAGGLE_LIMIT)}:t>) · training must end by <t:{int(DEADLINE)}:t>"]
    try:
        if quick:
            raise TimeoutError("skipped")
        rows = subprocess.run(["nvidia-smi", "--query-gpu=index,utilization.gpu,memory.used,memory.total,temperature.gpu",
                               "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=20).stdout
        for row in rows.strip().splitlines():
            index, busy, used, total, heat = [part.strip() for part in row.split(",")]
            lines.append(f"GPU {index}: {busy}% busy · {float(used) / 1024:.1f}/{float(total) / 1024:.0f} GB · {heat}°C")
    except Exception as exc:
        if not quick:
            lines.append(f"GPUs: unknown ({type(exc).__name__})")
    try:
        import psutil
        memory = psutil.virtual_memory()
        lines.append(f"RAM {memory.used / 1e9:.1f}/{memory.total / 1e9:.0f} GB ({memory.percent:.0f}%) · "
                     f"CPU {psutil.cpu_percent():.0f}% of {CPUS} cores")
    except Exception:
        pass
    lines.append(f"Disk free: /kaggle/working {shutil.disk_usage(WORK).free / 1e9:.1f} GB · "
                 f"/kaggle/tmp {shutil.disk_usage(TMP).free / 1e9:.0f} GB")
    return "\n".join(lines + edition_lines()[0])


def cliff_report(quick=False):
    """All the details for the last minutes before THE 12 HOURs: status + what is safe and what is not."""
    lines, safe = edition_lines()
    total = len(lines)
    verdict = (f"🌳 **The branch holds:** all {total} edition(s) are safe on Hugging Face." if total and safe == total
               else f"🌿 **Holding on:** {safe}/{total} edition(s) safe on Hugging Face - the rest is still on this "
                    "server; anything marked ⚠️ is lost with it (download it from the Output / file panel).")
    return server_status(quick) + "\n\n" + verdict


# ---- THE 12 HOURs: Kaggle kills every session 12 h after it started - the cliff this notebook refuses to fall off
CLIFF_MARKS = (300, 180, 120, 60, 30, 15, 10, 5, 4, 3, 2, 1, 0.5)  # minutes before THE 12 HOURs that get a message
CLIFF_PINGS = {300, 180, 120, 60, 30, 15, 10, 5, 1, 0.5}  # ... and ping you (4, 3 and 2 min come silently)
ROLE_CLIFF_MINUTES = 45  # the ping_role role hears THE 12 HOURs at the long marks only (5 h, 3 h, 2 h, 1 h)


def role_hears(title, ping):
    """Does the ping_role role get pinged for this server warning? Not for the last-hour countdown."""
    prefix = "THE 12 HOURs: "
    if not ping or not title.startswith(prefix):
        return bool(ping)
    try:
        number, unit = title[len(prefix):].removesuffix(" left").split()
        minutes = float(number) * {"h": 60.0, "min": 1.0, "s": 1 / 60}[unit]
    except (ValueError, KeyError):
        return True
    return minutes >= ROLE_CLIFF_MINUTES


def cliff_wind(mark, left):
    if mark >= 120:
        return (f"🌬️ The wind is rising - THE 12 HOURs is {mark // 60:.0f} hours away. Holding on; the work goes on, "
                "and everything gets to Hugging Face before the edge.")
    if mark >= 60:
        return ("🌪️ Strong winds now - 1 hour to THE 12 HOURs. I'm holding the branch tight: whatever isn't on Hugging "
                "Face by then falls with the server.")
    if mark >= 10:
        return f"🌪️ The gusts pull hard - {left:.0f} min to THE 12 HOURs. Both hands on the branch. Where things stand:"
    if mark >= 1:
        return (f"🌿 Knuckles white, still holding the branch - {max(left, 0):.0f} min to THE 12 HOURs. Everything, "
                "in case the wind wins:")
    return (f"🪢 Fingers slipping, not letting go - {max(left, 0) * 60:.0f} s to THE 12 HOURs. My final report - if "
            f"no message follows, THE 12 HOURs took the server at <t:{int(KAGGLE_LIMIT)}:t>:")


def cliff_label(left):
    if left >= 90:
        return f"{left / 60:.1f}".rstrip("0").rstrip(".") + " h"
    return f"{max(left, 0) * 60:.0f} s" if left < 1 else f"{left:.0f} min"


def server_warnings(now, warned):
    """Messages (title, text, ping) that are due now; `warned` (a dict) remembers what was already said and when."""
    due = []
    left = (KAGGLE_LIMIT - now) / 60
    marks = [mark for mark in CLIFF_MARKS if left <= mark and mark not in warned]
    if marks and not DISCORD_STATE.get("finished"):
        mark = min(marks)  # a late check sends only the newest one
        for passed in marks:
            warned[passed] = now
        details = cliff_report() if mark <= 60 else server_status()
        ping = mark in CLIFF_PINGS
        due.append((f"THE 12 HOURs: {cliff_label(left)} left", cliff_wind(mark, left) + "\n\n" + details, ping))
    try:
        free = shutil.disk_usage(WORK).free / 1e9
    except OSError:
        free = 99.0
    if free < 2 and now - warned.get("disk", 0.0) > 1800:
        warned["disk"] = now
        due.append(("Disk almost full", f"/kaggle/working has only {free:.1f} GB free (20 GB in total) - "
                    "exports and uploads may fail.", True))
    try:
        import psutil
        ram = psutil.virtual_memory().percent
    except Exception:
        ram = 0.0
    if ram > 93 and now - warned.get("ram", 0.0) > 1800:
        warned["ram"] = now
        due.append(("RAM almost full", f"RAM is {ram:.0f}% used - the server may kill the training.", False))
    return due


def server_monitor():
    warned, last = {}, time.time()
    while True:
        time.sleep(5 if KAGGLE_LIMIT - time.time() < 7 * 60 else 30)  # the last minutes: on time to the second
        try:
            for title, text, ping in server_warnings(time.time(), warned):
                discord(text, title, "bad" if title.startswith("THE 12") and "min" in title and float(
                    title.split()[3]) <= 10 else "warn", ping=ping, role=role_hears(title, ping))
                if title.startswith("THE 12"):
                    print(f"[THE 12 HOURs] {title}", flush=True)
            if DISCORD_EVERY_MIN and time.time() - last >= DISCORD_EVERY_MIN * 60:
                last = time.time()
                discord(server_status(), "Server status", "info")
        except Exception as exc:  # the monitor never stops the notebook
            print(f"WARN  Discord monitor: {type(exc).__name__}: {exc}", flush=True)


def server_closing(*_args):
    """The kernel is shutting down (stop session, restart, idle timeout): say so before it goes."""
    if not DISCORD_STATE.get("closing"):
        DISCORD_STATE["closing"] = True
        try:
            details = "\n\n" + cliff_report(quick=True)
        except Exception:
            details = ""
        if DISCORD_STATE.get("finished"):  # the last cell ran: an expected end
            discord(f"The Kaggle server is shutting down - the notebook finished.\n{SERVER_URL}{details}",
                    "Server closed", "ok")
        elif KAGGLE_LIMIT - time.time() < 15 * 60:
            discord(f"🏔️ THE 12 HOURs has come - Kaggle is closing the server. I held the branch to the last second; "
                    f"this is where everything stands:{details}", "Server closing", "bad", ping=True)
        else:
            discord(f"The Kaggle server is shutting down now (session stopped, restarted, timed out or out of "
                    f"time) before the notebook finished.\n{SERVER_URL}{details}", "Server closing", "bad", ping=True)
        discord_flush(8)


if DISCORD_URL:
    import atexit
    import signal
    atexit.register(server_closing)
    try:
        previous = signal.getsignal(signal.SIGTERM)

        def on_sigterm(number, frame):
            server_closing()
            signal.signal(signal.SIGTERM, previous)  # then shut down exactly as before
            os.kill(os.getpid(), signal.SIGTERM)

        signal.signal(signal.SIGTERM, on_sigterm)
    except (ValueError, OSError):  # not the main thread
        pass
    if not globals().get("SERVER_MONITOR"):  # one monitor, even when this cell runs again
        SERVER_MONITOR = threading.Thread(target=server_monitor, daemon=True)
        SERVER_MONITOR.start()
    discord(server_status() + f"\nStatus every {DISCORD_EVERY_MIN} min - if they stop coming, the server is gone."
            if DISCORD_EVERY_MIN else server_status(), "Kaggle server online", "ok")
# ---- end Discord server monitor


In [ ]:
# ============================== dry run (no GPU time wasted on broken data) ==============================
import shlex

READY = []
dry_runs = {edition: subprocess.Popen(["python", SCRIPT, "--preset", edition, "--dry-run"] + shlex.split(FLAGS),
                                      stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
            for edition in EDITIONS}  # all at once: CPU only, a few seconds instead of a few seconds each
for edition, check_run in dry_runs.items():
    stdout, stderr = check_run.communicate()
    print(f"===== {edition}\n{stdout[-3000:]}{stderr[-1500:]}")
    if check_run.returncode == 0:
        READY.append(edition)
    else:
        print(f"skipping {edition}: the dry run failed (see above)")
print("will train:", READY)
if not READY:
    discord("Every edition failed its dry run - nothing will train. Open the notebook to see why.",
            "Nothing to train", "bad", ping=True)


In [ ]:
# ============================== optional: Ollama for the GGUF test ==============================
if TEST_GGUF and not QUICK_TEST and not shutil.which("ollama"):
    !apt-get -qq update > /dev/null && apt-get -qq install -y zstd pciutils > /dev/null
    !curl -fsSL https://ollama.com/install.sh | sh > /dev/null
    print("Ollama:", shutil.which("ollama") or "install FAILED - the GGUF test will be skipped")

In [ ]:
# ============================== train on both T4s (+ tests, safetensors, GGUF quants, upload) ==============================
# GPU_MODE "merge": both T4s train every edition together (DDP via HF Accelerate); "parallel": one edition per T4.
# Each edition logs to /kaggle/working/<edition>.log (everything); here it is shown tidied, prefixed with the edition:
# no line per training step, no doubled banners from the two T4 processes, one status line with the ETA every ~5 min.
import collections
import time

# ---- log tidy
BAR_LINE = re.compile(r"(?P<label>[^|]*?):?\s*(?P<pct>\d+)%\|[^|]*\|\s*(?P<done>\d+)/(?P<total>\d+)\s*"
                      r"\[(?P<spent>[\d:]+)(?:<(?P<left>[\d:?]+))?(?:,\s*(?P<rate>[^\]]*))?\]")
LOSS_LINE = re.compile(r"\{'loss': '?([\d.e+-]+)'?.*?'epoch': '?([\d.]+)'?")
TIDY = {"bars": {}, "loss": {}, "recent": {}}


def clock_hours(text):
    """\"4:40:38\" / \"33:33\" -> hours (None for \"?\")."""
    try:
        parts = [int(part) for part in text.split(":")]
    except (AttributeError, ValueError):
        return None
    seconds = 0
    for part in parts:
        seconds = seconds * 60 + part
    return seconds / 3600


def tidy(edition, line, now=None):
    """A trainer line as shown here, None = not shown (it is still in the edition's log file)."""
    now = time.time() if now is None else now
    loss = LOSS_LINE.search(line)
    if loss:  # every 10 steps: goes into the next status line instead
        TIDY["loss"][edition] = (float(loss[1]), float(loss[2]))
        return None
    bar = BAR_LINE.search(line)
    if bar:
        label = bar["label"].strip() or "steps"
        done, total = int(bar["done"]), int(bar["total"])
        last = TIDY["bars"].get((edition, label))
        every = 300 if label == "steps" else 60
        if last and (last[1] == done or (done < total and now - last[0] < every) or last[1] >= total):
            return None  # the other T4's copy, or too soon
        TIDY["bars"][(edition, label)] = (now, done)
        after = line[bar.end():].strip()
        if label != "steps":
            return f"{label}: {done}/{total} ({bar['pct']}%)" + (f"  {after}" if after else "")
        text = f"step {done}/{total} ({bar['pct']}%)"
        if edition in TIDY["loss"]:
            text += " · loss {:.3f} · epoch {:.2f}".format(*TIDY["loss"][edition])
        if bar["rate"]:
            text += f" · {bar['rate'].strip()}"
        left, session = clock_hours(bar["left"]), (DEADLINE - now) / 3600
        if left is not None and done < total:
            text += f" · ~{left:.1f} h to go (session: {session:.1f} h left)"
            if left > session - 0.5:
                text += " - won't finish all steps: it stops in time to export + upload"
        return text + (f"  {after}" if after else "")
    recent = TIDY["recent"].setdefault(edition, collections.deque(maxlen=60))
    if (line in recent and line[:1].strip() and len(line.strip()) > 3
            and not line.startswith(("Traceback", "File", "RuntimeError", "teacher", "distillation"))):
        return None  # the second T4 process printing the same banner
    recent.append(line)
    return line
# ---- end log tidy

# ---- Discord events: (shown log line, level, seconds before the same kind of message may come again, ping, title)
DISCORD_EVENTS = [
    (re.compile(r"^===== \S+ on GPU"), "info", 0, False, ""),                       # an edition starts
    (re.compile(r"^loading \S+ \(.*\) on \d+ GPU"), "info", 600, True, "now training"),
    (re.compile(r"^distillation \(Qwen3 strong-to-weak\)"), "info", 60, True, "now distilling"),
    (re.compile(r"^on-policy distillation: \d+ steps"), "info", 60, True, "now distilling"),   # phase 2
    (re.compile(r"^teacher wrote \d+ answers|^distillation: \d+/\d+ rows"), "ok", 60, False, ""),
    (re.compile(r"^learned: "), "ok", 0, False, ""),                                 # STOP_WHEN_LEARNED stopped it
    (re.compile(r"^on-policy distillation: KL"), "ok", 0, False, ""),
    (re.compile(r"^making GGUFs|^exporting GGUF \("), "info", 0, True, "now quantizing"),
    (re.compile(r"^saving the safetensors edition"), "info", 0, True, "now uploading" if globals().get("HF_REPO")
     else "now exporting"),
    (re.compile(r"^uploaded the (safetensors|GGUF) edition|^\s*uploaded \S+\.gguf "), "ok", 0, True,
     "model update on Hugging Face"),
    (re.compile(r"^teacher: .*(did not run|did not finish|already failed)|^distillation: the teacher failed|"
                r"upload failed|export failed|out of GPU memory|training on both T4s failed|"
                r"^on-policy distillation (failed|stopped)|won't finish all steps"), "warn", 600, False, ""),
    (re.compile(r"^Traceback|^\S*(Error|Exception): "), "bad", 900, False, ""),
]
DISCORD_STATUS = {}  # edition -> its newest status line (step, loss, ETA)


def discord_line(edition, line):
    """Send a shown log line to Discord when it is one of the events above."""
    if line.startswith("step "):
        DISCORD_STATUS[edition] = line
    for number, (pattern, level, every, ping, what) in enumerate(DISCORD_EVENTS):
        if pattern.search(line):
            text = line.strip("= ").strip() if line.startswith("=====") else line.strip()
            return discord(text, f"{edition}: {what}" if what else edition, level, ping=ping, key=(edition, number),
                           every=every)
    return False

def discord_tail(edition, lines=15):
    """The end of an edition's log, for a failure message."""
    try:
        with open(f"/kaggle/working/{edition}.log", encoding="utf-8", errors="replace") as handle:
            tail = [line.split("\r")[-1].rstrip() for line in handle.readlines()[-lines:]]
    except OSError:
        return ""
    return "\n```\n" + "\n".join(tail)[-1500:].replace("```", "` ` `") + "\n```"

n_gpus = len(gpus)
MODE = {"merge": "ddp", "merged": "ddp"}.get(GPU_MODE, GPU_MODE) if n_gpus >= 2 else "single"
if MODE in ("auto", "parallel"):
    MODE = "parallel" if len(READY) >= 2 else "ddp"  # one edition: both T4s merge on it instead of one T4 idling
SLOTS = {"parallel": [[0], [1]], "ddp": [[0, 1]]}.get(MODE, [[0]])
RUN_FLAGS = FLAGS + f" --reserve-minutes {90 if MODE == 'parallel' else 60}"  # parallel: the other export may run first
if len(READY) > len(SLOTS) and not TEST_GGUF:
    # more editions than T4s: once an edition's safetensors are saved, its GGUF quants + uploads run in a CPU-only
    # process and the next edition starts on that T4 straight away (instead of ~20-40 min later)
    RUN_FLAGS += " --defer-gguf"
print(f"{n_gpus} GPU(s), mode {MODE}: " + {"parallel": "one edition per T4, two at a time",
                                           "ddp": "both T4s merged: they train each edition together (DDP)",
                                           "single": "one T4, one edition after another"}.get(MODE, MODE))
discord(f"Training **{', '.join(READY) or 'nothing'}** on {n_gpus} GPU(s), mode `{MODE}`"
        + (" (both T4s together)" if MODE == "ddp" else " (one edition per T4)" if MODE == "parallel" else "")
        + f".\nTraining ends by <t:{int(DEADLINE)}:t> (<t:{int(DEADLINE)}:R>) at the latest."
        + ("\nQUICK_TEST: a short pipeline check." if QUICK_TEST else ""), "Setup ready - training starts", "ok", ping=True)
if DISTILL and "VLLM_THREAD" in globals():
    if VLLM_THREAD.is_alive():
        wait = globals().get("VLLM_WAIT", 600)
        print(f"waiting for the vLLM install + teacher pre-flight ({wait / 60:.0f} min at most)...")
        VLLM_THREAD.join(wait)
    if VLLM_THREAD.is_alive() and "VLLM_ABORT" in globals():  # too slow: stop it (frees the T4s), train without it
        VLLM_ABORT.set()
        VLLM_THREAD.join(60)
    if VLLM_READY.is_set():
        RUN_FLAGS += f' --vllm-python "{VLLM_PYTHON}"'
        print("teacher: vLLM on both T4s")
    else:
        why = globals().get("VLLM_STATE", {}).get("cause") or "see /kaggle/working/vllm-install.log"
        print(f"WARN  vLLM is not ready ({why.splitlines()[-1][:200]})")
        if not globals().get("SLOW_TEACHER", True):
            RUN_FLAGS += " --distill-minutes 0"  # cached answers still count; no slow writing with transformers
            print("      training starts at once: no new phase-1 answers with transformers (SLOW_TEACHER = False); "
                  "cached ones are used, phase 2 still runs")
        else:
            print("      the teacher writes with transformers instead (slower)")
if MODE == "parallel" and not HF_REPO and SAFETENSORS:
    print("WARN  two editions without a Hugging Face upload can pass Kaggle's 20 GB output limit "
          "(~11 GB each) - add HF_TOKEN, set SAFETENSORS = False, or train one edition per session")


def train_command(edition, gpu_ids, extra=""):
    script = [SCRIPT, "--preset", edition] + shlex.split(RUN_FLAGS + " " + extra)
    if len(gpu_ids) > 1:  # DDP: one process per T4, gradients synced after every step
        if importlib.util.find_spec("accelerate"):  # HF Accelerate: multi-GPU launch with fp16 mixed precision
            return [sys.executable, "-m", "accelerate.commands.launch", "--multi_gpu", f"--num_processes={len(gpu_ids)}",
                    "--num_machines=1", "--mixed_precision=fp16", "--dynamo_backend=no",
                    f"--main_process_port={29500 + len(CODES)}"] + script
        return [sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={len(gpu_ids)}"] + script
    return [sys.executable] + script


def run_edition(edition, gpu_ids, extra=""):
    """Train one edition; returns (exit code, its log output)."""
    path = f"/kaggle/working/{edition}.log"
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": ",".join(map(str, gpu_ids)), "PYTHONUNBUFFERED": "1",
           "PYMC_RUN_ID": f"{edition}-{time.time_ns()}"}  # the two T4 processes find each other's hand-over files
    with open(path, "a", encoding="utf-8") as log:
        log.write(f"===== {edition} on GPU {','.join(map(str, gpu_ids))} =====\n")
        log.flush()
        start = log.tell()
        code = subprocess.run(train_command(edition, gpu_ids, extra), stdout=log, stderr=subprocess.STDOUT,
                              env=env, cwd=f"{TMP}/run").returncode  # unsloth_compiled_cache etc. stay out of WORK
    with open(path, encoding="utf-8", errors="replace") as log:
        log.seek(start)
        return code, log.read()


def export_job(edition):
    return f"/kaggle/working/{edition}/export_job.json"


def run_export(edition):
    """GGUF quants + uploads of an edition trained with --defer-gguf: CPU and network only, no T4."""
    progress.pop(edition, None)
    env = {**os.environ, "CUDA_VISIBLE_DEVICES": "", "PYTHONUNBUFFERED": "1"}
    with open(f"/kaggle/working/{edition}.log", "a", encoding="utf-8") as log:
        log.write(f"===== {edition}: GGUF quants + upload (CPU, while the T4 trains the next edition) =====\n")
        log.flush()
        code = subprocess.run([sys.executable, SCRIPT, "--export-only", f"/kaggle/working/{edition}"], stdout=log,
                              stderr=subprocess.STDOUT, env=env, cwd=f"{TMP}/run").returncode
    CODES[edition] = code
    clean_up(edition)


def note(edition, text):
    """A line in the edition's log, so it shows up here in order with the trainer's own output."""
    with open(f"/kaggle/working/{edition}.log", "a", encoding="utf-8") as log:
        log.write(text + "\n")


def report_of(edition):
    path = f"/kaggle/working/{edition}/pymc_model.json"
    return json.load(open(path)) if os.path.exists(path) else {}


def clean_up(edition):
    info = report_of(edition)
    if CLEAN_AFTER_UPLOAD and info.get("hub_repo"):
        # GGUF repo is up: drop the GGUF files; the safetensors folder only once its own upload worked too
        big = glob.glob(f"/kaggle/working/{edition}/*.gguf") + [f"/kaggle/working/{edition}/lora"]
        if info.get("safetensors_uploaded") or not info.get("safetensors"):
            big.append(f"/kaggle/working/{edition}/safetensors")
        for path in big:
            if os.path.isdir(path):
                shutil.rmtree(path, ignore_errors=True)
            elif os.path.exists(path):
                os.remove(path)
        note(edition, "uploaded, big local files cleaned up")
    if info.get("safetensors") and not info.get("safetensors_uploaded"):
        note(edition, f"the safetensors edition is in /kaggle/working/{edition}/safetensors (not uploaded)")


queue, queue_lock, CODES, EXPORTS, progress = list(READY), threading.Lock(), {}, [], {}


def failed(code, edition):
    return code not in (0, 3) and not report_of(edition) and not os.path.exists(export_job(edition))


BUSY = set()  # parallel: the T4 slots training right now


def worker(gpu_ids):
    while True:
        with queue_lock:
            if not queue:
                return
            edition = queue.pop(0)
            used = gpu_ids
            if MODE == "parallel" and not queue and not BUSY and n_gpus >= 2:
                used = list(range(n_gpus))  # last edition and the other T4 is free: both T4s merge on it
            BUSY.add(tuple(gpu_ids))
        try:
            train_slot(edition, gpu_ids, used)
        finally:
            with queue_lock:
                BUSY.discard(tuple(gpu_ids))


def train_slot(edition, gpu_ids, used):
    code, output = run_edition(edition, used)
    if failed(code, edition) and len(used) > 1:
        note(edition, f"training on both T4s failed (exit {code}) - training it again on one T4")
        used = gpu_ids[:1]
        code, output = run_edition(edition, used)
    if failed(code, edition) and "out of memory" in output.lower() and FAST:
        note(edition, "ran out of GPU memory with FAST's bigger batches - training it again with the preset's batch")
        code, output = run_edition(edition, used, "--token-budget 0")
    if code == 0 and os.path.exists(export_job(edition)):
        exporter = threading.Thread(target=run_export, args=(edition,), daemon=True)
        exporter.start()
        EXPORTS.append(exporter)  # this T4 takes the next edition now
        return
    CODES[edition] = code
    clean_up(edition)


threads = [threading.Thread(target=worker, args=(slot,), daemon=True) for slot in SLOTS]
for thread in threads:
    thread.start()
offsets, partial, last_beat = {}, {}, time.time()
while True:
    alive = any(thread.is_alive() for thread in threads + list(EXPORTS))
    for edition in READY:
        path = f"/kaggle/working/{edition}.log"
        if not os.path.exists(path):
            continue
        with open(path, encoding="utf-8", errors="replace") as handle:
            handle.seek(offsets.get(edition, 0))
            chunk = handle.read()
            offsets[edition] = handle.tell()
        text = partial.get(edition, "") + chunk
        *lines, partial[edition] = text.split("\n")
        for line in lines:
            line = tidy(edition, line.split("\r")[-1].rstrip())
            if line:
                print(f"[{edition}] {line}", flush=True)
                discord_line(edition, line)
        if "\r" in partial[edition]:  # a progress bar: shown every few minutes, not on every step
            progress[edition] = partial[edition].split("\r")[-1].strip() or progress.get(edition, "")
    if time.time() - last_beat > 180:
        last_beat = time.time()
        left = (DEADLINE - time.time()) / 3600
        for edition, bar in progress.items():
            if bar and edition not in CODES:
                print(f"[{edition}] {bar}   (session: {left:.1f} h left)", flush=True)
    if not alive:
        break
    time.sleep(5)
for edition in READY:
    code = CODES.get(edition)
    print(f"{edition}: " + {0: "done", 3: "skipped - not enough session time left (train it next session)"}.get(
        code, f"FAILED (exit {code}) - see /kaggle/working/{edition}.log"))
    if code == 3:
        discord("Not enough session time left - train it in the next session.", f"{edition}: skipped", "warn")
    elif code != 0:
        discord(f"Exit code {code}. End of /kaggle/working/{edition}.log:" + discord_tail(edition),
                f"{edition}: FAILED", "bad", ping=True)


In [ ]:
# ============================== results ==============================
for edition in READY:
    folder = f"/kaggle/working/{edition}"
    if not os.path.exists(f"{folder}/pymc_model.json"):
        print(f"{edition}: no report - look at the training output above")
        discord("No report was written - look at the training output on Kaggle.", f"{edition}: no result", "bad")
        continue
    info = json.load(open(f"{folder}/pymc_model.json"))
    where = f"https://huggingface.co/{info['hub_repo']}" if info.get("hub_repo") else folder
    print(f"\n### {info.get('model_name', edition)} -> {where}")
    print(f"loss {info.get('loss')} | {info.get('train_rows')} rows ({info.get('general_rows', 0)} everyday) | "
          f"{info.get('minutes')} min | eval {info.get('eval', {})}")
    if info.get("stopped_early"):
        print(f"stopped early at step {info['stopped_early']['step']}/{info['stopped_early']['of']} (session time limit)")
    if info.get("tests"):
        print("tests:", info["tests"])
    if info.get("gguf_test"):
        print("GGUF test:", "ok" if info["gguf_test"].get("ok") else info["gguf_test"])
    if info.get("safetensors_uploaded"):
        print(f"safetensors ({info.get('safetensors_gb')} GB): https://huggingface.co/{info['hub_repo_safetensors']}")
    elif info.get("safetensors"):
        print(f"safetensors ({info.get('safetensors_gb')} GB): {folder}/safetensors")
    if info.get("ggufs"):
        print("GGUFs:", ", ".join(f"{g['quant']} {g['gb']} GB" + ("" if g.get("uploaded") or g["quant"] == info.get("quant")
                                                                   else " (local)") for g in info["ggufs"]))
    if info.get("params_b"):
        print(f"parameters: {info['params_b']}B (compact: {info.get('compact', {}).get('tied_dropped') or 'nothing to drop'})")
    if info.get("hub_repo"):
        print(f"on your PC:  python -m pymc_bot llm import --hf {info['hub_repo']}  (add --quant Q8_0 for the big one)")
    summary = [f"**Where:** {where}", f"**Loss** {info.get('loss')} · {info.get('train_rows')} rows · "
               f"{info.get('minutes')} min"]
    if info.get("stopped_early"):
        summary.append(f"stopped at step {info['stopped_early']['step']}/{info['stopped_early']['of']} (session time)")
    if info.get("safetensors_uploaded"):
        summary.append(f"**safetensors** ({info.get('safetensors_gb')} GB): "
                       f"https://huggingface.co/{info['hub_repo_safetensors']}")
    if info.get("ggufs"):
        summary.append("**GGUFs:** " + ", ".join(f"{g['quant']} {g['gb']} GB" for g in info["ggufs"]))
    if info.get("tests"):
        summary.append(f"tests: {str(info['tests'])[:600]}")
    if info.get("hub_repo"):
        summary.append(f"`python -m pymc_bot llm import --hf {info['hub_repo']}`")
    discord("\n".join(summary), f"{info.get('model_name', edition)} is ready", "ok", ping=True)

## Optional: test again / chat with a trained edition

Needs the edition's `lora` folder in `/kaggle/working/<edition>` (set `CLEAN_AFTER_UPLOAD = False` before training to keep it).

In [ ]:
RETEST = []   # e.g. ["minecraftism"]: run the test suite again on the saved adapter
for edition in RETEST:  # noqa: B007 - used by the ! line
    !python "{SCRIPT}" --preset {edition} --test-only {FLAGS}

In [ ]:
CHAT_WITH = ""   # e.g. "minecraftism": chat with a text edition right here (empty line = stop). Not for "Save & Run All".
if CHAT_WITH:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(f"/kaggle/working/{CHAT_WITH}/lora", max_seq_length=4096,
                                                         load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    history = [{"role": "system", "content": SYSTEM_PROMPT.strip()}]
    while True:
        line = input("you> ").strip()
        if not line:
            break
        history.append({"role": "user", "content": line})
        prompt = tokenizer.apply_chat_template(history, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text=prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
        output = model.generate(**inputs, max_new_tokens=1024, temperature=0.7, top_p=0.8, top_k=20, do_sample=True)
        reply = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        print(f"minecraftism> {reply}\n")
        history.append({"role": "assistant", "content": reply.split("</think>")[-1].strip()})
    del model
    import torch
    torch.cuda.empty_cache()

## Next
* From Hugging Face: `python -m pymc_bot llm import --hf <your-hf-name>/Minecraftism-4B-Uncensored-GGUF`
  (creates the Ollama model with this system prompt, e.g. `minecraftism-uncensored`, and switches the bot to it).
  Or download `/kaggle/working/<edition>` from the Output tab and run `python -m pymc_bot llm import <folder>`.
* The **safetensors** edition (`<your-hf-name>/Minecraftism-4B-Uncensored`, no `-GGUF`) is for transformers / vLLM
  (`vllm serve <your-hf-name>/Minecraftism-4B-Uncensored`), further fine-tuning or your own quantisations; its model
  card has the code. `llm import --hf` on it uses the GGUF twin automatically.
* Other GGUF sizes from the same repo: `ollama run hf.co/<your-hf-name>/Minecraftism-4B-Uncensored-GGUF:Q8_0`
  (or any quant in the repo's table: `:Q6_K`, `:IQ4_XS`, `:IQ3_M`...); `llm import --hf ... --quant Q8_0` does the same for the bot.
* As a daily assistant without the bot: `ollama run minecraftism-uncensored` (or any Ollama app such as Open WebUI).
* VL editions also turn on **let the model see** in the bot (optional vision packages, see docs/MINECRAFT_LLM.md).
* Compare: `python -m pymc_bot llm eval --model minecraftism-uncensored` vs `--model qwen3:4b`.

In [ ]:
# ============================== auto exit: end the session when everything is done ==============================
# Kaggle has no "stop session" call, so:
# * Save Version -> "Save & Run All" (recommended for long runs): the session ends by itself after this last cell
#   and /kaggle/working is kept as the notebook's Output - nothing to do here.
# * Interactive editor: the session would idle on (and use GPU quota) until Kaggle's idle timeout. When every
#   trained edition is safely on Hugging Face, this stops the kernel and its processes; files that only exist in
#   /kaggle/working are never thrown away like that (they vanish with an interactive session).
run_type = os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "")
DISCORD_STATE["finished"] = True  # the server closing after this is expected
reports = {edition: report_of(edition) for edition in READY}
on_hub = bool(reports) and all(info.get("hub_repo") and (info.get("safetensors_uploaded") or not info.get("safetensors"))
                               for info in reports.values())
if not AUTO_EXIT:
    print("AUTO_EXIT = False - remember to stop the session (Run > Stop session) to save GPU quota")
    discord("All cells finished. AUTO_EXIT is off - stop the session (Run > Stop session) to save GPU quota.\n\n"
            + cliff_report(), "Notebook finished", "warn", ping=True)
elif run_type == "Batch":
    print("Save & Run All: the session ends right after this cell; /kaggle/working is saved as the Output")
    discord("Save & Run All finished - the session ends now; /kaggle/working is kept as the notebook's Output.\n\n"
            + cliff_report(), "Session finished", "ok", ping=True)
elif on_hub:
    print("everything is on Hugging Face - ending this session now so no GPU time is wasted")
    print("(if the editor still shows it as running, press Run > Stop session)", flush=True)
    discord("Everything is on Hugging Face - ending the Kaggle session now, so no GPU quota is wasted.\n\n"
            + cliff_report(), "Session finished", "ok", ping=True)
    discord_flush()
    time.sleep(5)
    subprocess.Popen(["bash", "-c", "sleep 1; kill -9 -1"], start_new_session=True)  # every process of this session
    os._exit(0)
else:
    print("not ending the session: some files are only in /kaggle/working (no upload, or an upload failed) - "
          "download them from the Output / file panel, then press Run > Stop session")
    discord("Some files are only in /kaggle/working (no upload, or an upload failed). Download them from the Output "
            "/ file panel, then press Run > Stop session - it is still using GPU quota.\n\n" + cliff_report(),
            "Session still running", "warn", ping=True)
discord_flush()
